# Semana 6 · Limpieza de datos — Evaluación Fitosanitaria (Arañita Roja)
## Inspección, limpieza y transformación básica de datos

**Objetivo técnico:** construir una versión limpia y trazable del histórico de monitoreo fitosanitario, sin inventar información ni borrar automáticamente observaciones que podrían ser válidas.

**Producto:** `fitosanitario_limpio.csv`

> Regla central: el archivo original se conserva intacto. Todo tratamiento se ejecuta sobre una copia y deja evidencia.

## Caso
El fundo registra evaluaciones fitosanitarias de campo (formulario) por módulo y válvula de riego, con el nivel de infestación de arañita roja (*Oligonychus punicae*) en dos indicadores: hoja y postura (huevos).

**Campos principales:** marca temporal, campaña, módulo, turno, válvula, plantas evaluadas, % hojas infestadas, grado hoja, % infestación postura, grado postura.

**Regla de negocio confirmada con el equipo del fundo:** en las columnas de arañita, una celda vacía **no significa dato faltante** — significa que en esa evaluación no se encontró infestación, es decir, el valor real es **0**. Esta es una decisión distinta a la de un dataset genérico: aquí sí existe una regla defendible para "rellenar", porque no es una suposición nuestra sino la convención con la que el propio fundo llena el formulario.

In [ ]:
# Si estás en Google Colab, esto abre el selector de archivos:
# sube el archivo Fitosanitario_sucio.xlsx cuando te lo pida (no importa el nombre exacto).
try:
    from google.colab import files
    uploaded = files.upload()
    ARCHIVO = next(iter(uploaded))
except ImportError:
    # Si NO estás en Colab (Jupyter local), coloca el archivo en la misma carpeta
    # que este notebook y ajusta el nombre aquí:
    ARCHIVO = "Fitosanitario_sucio.xlsx"

print("Archivo:", ARCHIVO)

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

df_raw = pd.read_excel(ARCHIVO, sheet_name="Respuestas de formulario 1")

print("Dimensiones recibidas:", df_raw.shape)
display(df_raw.head(8))

## 1. Preservar el original
No trabajaremos directamente sobre `df_raw`. Creamos una copia de trabajo para que cualquier cambio sea reversible y auditable.

In [ ]:
df = df_raw.copy()

print("¿El original y la copia tienen las mismas dimensiones?", df_raw.shape == df.shape)

## 2. Inspección estructural antes de limpiar
Un perfilado inicial debe responder, como mínimo:

- ¿Cuántas filas y columnas existen?
- ¿Cómo se llaman los campos?
- ¿Qué tipos detecta Python?
- ¿Qué columnas numéricas llegaron como texto (posibles errores de tipeo)?
- ¿Qué campañas, módulos y válvulas existen?

In [ ]:
print("FILAS:", df.shape[0])
print("COLUMNAS:", df.shape[1])

print("\nENCABEZADOS:")
print(df.columns.tolist())

print("\nTIPOS DETECTADOS:")
print(df.dtypes)

print("\nCAMPAÑAS:", df["CAMPAÑA"].unique())
print("MÓDULOS:", sorted(df["MODULO"].unique()))
print("TURNOS:", sorted(df["TURNO"].unique()))
print("N° VÁLVULAS distintas:", df["VALVULA"].nunique())

## 3. Normalizar los nombres de columnas
Usaremos nombres consistentes en minúsculas, sin tildes, espacios ni símbolos — esto no cambia los datos, solo la forma de referirnos a ellos.

In [ ]:
def normalizar_nombre_columna(nombre):
    nombre = str(nombre).strip().lower()
    nombre = "".join(
        ch for ch in unicodedata.normalize("NFKD", nombre)
        if not unicodedata.combining(ch)
    )
    nombre = re.sub(r"[^a-z0-9]+", "_", nombre)
    return nombre.strip("_")

df.columns = [normalizar_nombre_columna(c) for c in df.columns]

print(df.columns.tolist())

## 4. Convertir la fecha de forma controlada
`marca_temporal` trae fecha y hora exacta del registro. Para cruzar con el clima diario, solo necesitamos la fecha (sin hora).

In [ ]:
df["marca_temporal"] = pd.to_datetime(df["marca_temporal"], errors="coerce")
df["fecha"] = df["marca_temporal"].dt.normalize()

print("Fechas no interpretables:", df["fecha"].isna().sum())
print("Rango:", df["fecha"].min().date(), "->", df["fecha"].max().date())

## 5. Tratamiento de vacíos: NO son datos faltantes, son ceros
A diferencia de un dataset genérico (donde nunca se rellena un vacío sin una regla defendible), aquí sí tenemos esa regla, confirmada con el equipo del fundo: un campo vacío en las columnas de arañita significa que no se encontró infestación.

Por eso convertimos estas columnas a numérico y rellenamos los vacíos con 0 — documentando la decisión en la bitácora, no aplicándola "por conveniencia".

In [ ]:
columnas_arana = [
    "aranita_hojas_infestadas",
    "aranita_grado_hoja",
    "aranita_infestacion_postura",
    "aranita_grado_postura",
]
# Ajuste por si el normalizador produjo nombres ligeramente distintos:
columnas_arana = [c for c in df.columns if "aranita" in c]
print("Columnas de arañita detectadas:", columnas_arana)

vacios_antes = df[columnas_arana].isna().sum()
display(vacios_antes.to_frame("vacios_antes_de_rellenar"))

## 6. Convertir números que llegaron como texto
Antes de rellenar con 0, algunas celdas tienen errores de tipeo detectados en la inspección: un espacio en vez de un punto decimal (`"1 78"` en vez de `"1.78"`, `"0. 5"` en vez de `"0.5"`). Los corregimos explícitamente — no se descartan ni se "adivinan" al azar, se corrige el patrón de error identificado.

In [ ]:
correcciones_tipeo = {"1 78": "1.78", "0. 5": "0.5"}

for col in columnas_arana:
    df[col] = df[col].replace(correcciones_tipeo)
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Ahora sí, vacíos -> 0 (regla de negocio confirmada, no una suposición nuestra).
for col in columnas_arana:
    df[col] = df[col].fillna(0)

print("Vacíos restantes tras rellenar:")
print(df[columnas_arana].isna().sum())

## 7. Duplicados exactos vs. lógicos — detectar y resolver
- **Duplicado exacto:** la fila completa se repite. Se retira, quedándonos con la primera aparición.
- **Duplicado lógico:** misma fecha + módulo + válvula + turno, con la misma o distinta infestación registrada — es una evaluación cargada más de una vez en el formulario.

**Regla de resolución de duplicados lógicos:** se conserva la carga con la marca_temporal más reciente (la versión final/corregida del formulario) y se retiran las anteriores. No se promedia ni se descarta al azar — se documenta la regla y se deja un flag de trazabilidad en el registro que sobrevive.

In [ ]:
duplicados_exactos = df.duplicated(keep="first")
print("Duplicados exactos:", duplicados_exactos.sum())

df = df.loc[~duplicados_exactos].copy()

clave_negocio = ["fecha", "modulo", "valvula", "turno"]

grupo_tiene_duplicado_antes = df.duplicated(subset=clave_negocio, keep=False)
n_filas_en_grupos_duplicados = int(grupo_tiene_duplicado_antes.sum())
print("Filas involucradas en duplicados lógicos (antes de resolver):", n_filas_en_grupos_duplicados)

# Se conserva la carga con marca_temporal más reciente por cada evaluación repetida.
df = df.sort_values("marca_temporal")
tiene_duplicado = df.duplicated(subset=clave_negocio, keep=False)
es_la_mas_reciente = ~df.duplicated(subset=clave_negocio, keep="last")

df["flag_duplicado_logico_corregido"] = tiene_duplicado & es_la_mas_reciente

filas_a_retirar = tiene_duplicado & ~es_la_mas_reciente
n_retiradas = int(filas_a_retirar.sum())

df = df.loc[~filas_a_retirar].sort_values("fecha").reset_index(drop=True)

print("Registros retirados por duplicado lógico (se conservó la carga más reciente):", n_retiradas)
print("Evaluaciones marcadas como corregidas (sobrevivieron a un duplicado):", int(df["flag_duplicado_logico_corregido"].sum()))
print("Filas después de resolver duplicados:", len(df))

## 8. Valores imposibles: marcar antes de corregir
Reglas conocidas del propio formulario:

- `% hojas infestadas` y `% infestación postura` deben estar entre 0 y 100.
- `grado hoja` y `grado postura` son un índice de severidad que normalmente va de 0 a 4.

Un valor fuera de rango **no se borra ni se reemplaza por un número inventado**: se marca con un flag para verificar con el ingeniero de campo.

In [ ]:
df["flag_pct_hoja_imposible"] = (df["aranita_hojas_infestadas"] < 0) | (df["aranita_hojas_infestadas"] > 100)
df["flag_pct_postura_imposible"] = (df["aranita_infestacion_postura"] < 0) | (df["aranita_infestacion_postura"] > 100)
df["flag_grado_hoja_atipico"] = df["aranita_grado_hoja"] > 4
df["flag_grado_postura_atipico"] = df["aranita_grado_postura"] > 4

columnas_flags = [
    "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
    "flag_grado_hoja_atipico", "flag_grado_postura_atipico",
]
display(df[columnas_flags].sum().to_frame("registros_marcados"))

display(
    df.loc[df["flag_grado_postura_atipico"],
           ["fecha", "modulo", "valvula", "aranita_grado_postura"]]
)

## 9. Posibles atípicos en % de infestación: detectar no es eliminar
Aplicamos el criterio de percentiles como **bandera de contexto**, no como sentencia de borrado. En este proyecto, a diferencia de un dataset de ventas, un valor muy alto de infestación **no es ruido — es exactamente la señal de descontrol que queremos anticipar.** Por eso aquí no se marcan para descartar, solo para caracterizar el fenómeno.

In [ ]:
percentiles = df.loc[df["aranita_hojas_infestadas"] > 0, "aranita_hojas_infestadas"].quantile([.5, .75, .85, .9, .95, .99])
display(percentiles.to_frame("pct_hojas_infestadas"))

top_episodios = df.nlargest(10, "aranita_hojas_infestadas")[
    ["fecha", "modulo", "valvula", "aranita_hojas_infestadas"]
]
display(top_episodios)

## 10. Bitácora de limpieza
Documentamos qué problema se encontró, qué regla se aplicó y cuántos registros fueron afectados.

In [ ]:
bitacora = pd.DataFrame([
    {
        "etapa": "Estructura",
        "variable": "Encabezados",
        "problema": "Nombres con tildes, espacios y símbolos",
        "accion": "Normalizar a snake_case",
        "registros_afectados": len(df.columns),
        "justificacion": "Facilita un procesamiento reproducible.",
    },
    {
        "etapa": "Ausencias",
        "variable": "Columnas de arañita",
        "problema": "Celdas vacías",
        "accion": "Rellenar con 0",
        "registros_afectados": int(vacios_antes.sum()),
        "justificacion": "Regla de negocio confirmada: vacío = no se encontró infestación, no dato faltante.",
    },
    {
        "etapa": "Formato numérico",
        "variable": "grado_hoja, grado_postura",
        "problema": 'Errores de tipeo ("1 78", "0. 5")',
        "accion": "Corregir patrón y convertir a numérico",
        "registros_afectados": len(correcciones_tipeo),
        "justificacion": "Patrón de error identificado explícitamente, no una suposición.",
    },
    {
        "etapa": "Duplicados exactos",
        "variable": "Todas las columnas",
        "problema": "Filas completamente repetidas",
        "accion": "Conservar la primera aparición",
        "registros_afectados": int(duplicados_exactos.sum()),
        "justificacion": "Evita doble conteo de una misma evaluación.",
    },
    {
        "etapa": "Duplicados lógicos",
        "variable": "fecha, modulo, valvula, turno",
        "problema": "Misma evaluación cargada más de una vez (formulario reenviado o corregido)",
        "accion": "Conservar la carga con marca_temporal más reciente; retirar las anteriores",
        "registros_afectados": n_retiradas,
        "justificacion": "La carga más reciente representa la versión final/corregida del formulario; no se promedia ni se descarta al azar.",
    },
    {
        "etapa": "Valores imposibles",
        "variable": "grado_postura",
        "problema": "2 valores fuera de la escala normal 0-4 (10 y 5.35)",
        "accion": "Marcar con flag, pendiente de confirmar con el Ing. Kiyoshi",
        "registros_afectados": int(df["flag_grado_postura_atipico"].sum()),
        "justificacion": "No se descarta ni corrige sin verificación de campo.",
    },
])

display(bitacora)

## 11. Resumen antes de exportar

In [ ]:
resumen = pd.DataFrame({
    "indicador": [
        "Filas recibidas",
        "Filas después de limpieza",
        "Duplicados exactos eliminados",
        "Duplicados lógicos resueltos (registros retirados)",
        "Evaluaciones marcadas como corregidas",
        "Vacíos rellenados con 0",
        "Valores atípicos de grado_postura marcados",
    ],
    "resultado": [
        len(df_raw),
        len(df),
        int(duplicados_exactos.sum()),
        n_retiradas,
        int(df["flag_duplicado_logico_corregido"].sum()),
        int(vacios_antes.sum()),
        int(df["flag_grado_postura_atipico"].sum()),
    ],
})
display(resumen)

## 12. Exportar el producto
Se genera el dataset fitosanitario limpio, listo para cruzarse con el clima.

In [ ]:
df.to_csv("fitosanitario_limpio.csv", index=False, encoding="utf-8-sig")
bitacora.to_csv("bitacora_limpieza_fitosanitario.csv", index=False, encoding="utf-8-sig")

print("Archivos generados:")
print(" - fitosanitario_limpio.csv", df.shape)
print(" - bitacora_limpieza_fitosanitario.csv", bitacora.shape)

## Referencias técnicas
- pandas documentation — Missing data, `to_datetime`, `to_numeric`, `duplicated`.
- Plan de Trabajo, Proyecto Productivo IB — Sistema de alerta temprana basado en grados-día para el control de la arañita roja en palto Hass, Fundo Loma Verde S.A.C.